In [1]:
# ──── 1. 라이브러리 import ────────────────────────────────────────
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from tqdm import trange

# ──── 2. VGG 블록 & 모델 정의 ────────────────────────────────────
def conv_2_block(in_dim, out_dim):
    return nn.Sequential(
        nn.Conv2d(in_dim, out_dim, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2)
    )

def conv_3_block(in_dim, out_dim):
    return nn.Sequential(
        nn.Conv2d(in_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2)
    )

class VGG16_CIFAR(nn.Module):
    def __init__(self, base_dim=64, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_2_block(3, base_dim),               
            conv_2_block(base_dim, base_dim*2),      
            conv_3_block(base_dim*2, base_dim*4),    
            conv_3_block(base_dim*4, base_dim*8),    
            conv_3_block(base_dim*8, base_dim*8),    
        )
        self.classifier = nn.Sequential(
            nn.Linear(base_dim*8*1*1, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 1000),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1000, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# ──── 3. 설정 ───────────────────────────────────────────────────
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size    = 100
learning_rate = 2e-4
num_epochs    = 20

# ──── 4. 데이터 준비 & train/val split ──────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),
                         (0.2470,0.2435,0.2616))
])

# 전체 train+val 데이터셋
full_train_ds = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_ds = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

# 90% train / 10% val 로 분할
total_size = len(full_train_ds)
val_size   = int(total_size * 0.1)
train_size = total_size - val_size
train_ds, val_ds = random_split(full_train_ds, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=4)

# ──── 5. 모델·손실·옵티마이저·TensorBoard 준비 ─────────────────
model     = VGG16_CIFAR(base_dim=64, num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
writer    = SummaryWriter(log_dir="runs/cifar10_vgg")

# ──── 6. 학습 루프 ─────────────────────────────────────────────
global_step = 0
for epoch in range(1, num_epochs+1):
    ### (1) Train 단계
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        global_step  += 1

        if global_step % 50 == 0:
            writer.add_scalar("Train/Batch_Loss", loss.item(), global_step)

    epoch_loss = running_loss / len(train_loader)
    writer.add_scalar("Train/Epoch_Loss", epoch_loss, epoch)

    ### (2) Validation 단계
    model.eval()
    val_loss    = 0.0
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc   = val_correct / len(val_ds)

    writer.add_scalar("Val/Loss", val_loss, epoch)
    writer.add_scalar("Val/Acc",  val_acc,   epoch)

    print(f"[Epoch {epoch:02d}/{num_epochs}] "
          f"Train Loss: {epoch_loss:.4f}  Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

# ──── 7. Test(최종 평가) ────────────────────────────────────────
model.eval()
test_correct = 0
test_total   = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds   = outputs.argmax(1)
        test_correct += (preds == labels).sum().item()
        test_total   += labels.size(0)

print(f">>> Final Test Accuracy: {test_correct / test_total:.4f}")

# ──── 8. 마무리 ────────────────────────────────────────────────
writer.close()


Files already downloaded and verified
Files already downloaded and verified
[Epoch 01/20] Train Loss: 2.0190  Val Loss: 1.8051  Val Acc: 0.2390
[Epoch 02/20] Train Loss: 1.7260  Val Loss: 1.5714  Val Acc: 0.3794
[Epoch 03/20] Train Loss: 1.4028  Val Loss: 1.2721  Val Acc: 0.5320
[Epoch 04/20] Train Loss: 1.1689  Val Loss: 1.1021  Val Acc: 0.6040
[Epoch 05/20] Train Loss: 0.9692  Val Loss: 1.0154  Val Acc: 0.6316
[Epoch 06/20] Train Loss: 0.8132  Val Loss: 0.8302  Val Acc: 0.7128
[Epoch 07/20] Train Loss: 0.6726  Val Loss: 0.7649  Val Acc: 0.7398
[Epoch 08/20] Train Loss: 0.5675  Val Loss: 0.7212  Val Acc: 0.7684
[Epoch 09/20] Train Loss: 0.4711  Val Loss: 0.7173  Val Acc: 0.7702
[Epoch 10/20] Train Loss: 0.3884  Val Loss: 0.7102  Val Acc: 0.7736
[Epoch 11/20] Train Loss: 0.3227  Val Loss: 0.7654  Val Acc: 0.7772
[Epoch 12/20] Train Loss: 0.2667  Val Loss: 0.7983  Val Acc: 0.7858
[Epoch 13/20] Train Loss: 0.2192  Val Loss: 0.8148  Val Acc: 0.7872
[Epoch 14/20] Train Loss: 0.1841  Val Lo

![](https://velog.velcdn.com/images/changh2_00/post/ebb3dfc2-5f7f-4b59-bf6a-181fb0726e70/image.png)

In [ ]:
# ──── 1. 라이브러리 import ────────────────────────────────────────
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from tqdm import trange

# ──── 2. VGG 블록 & 모델 정의 ────────────────────────────────────
def conv_2_block(in_dim, out_dim):
    return nn.Sequential(
        nn.Conv2d(in_dim, out_dim, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2)
    )

def conv_3_block(in_dim, out_dim):
    return nn.Sequential(
        nn.Conv2d(in_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_dim, out_dim, 3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2)
    )

class VGG16_CIFAR(nn.Module):
    def __init__(self, base_dim=64, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_2_block(3, base_dim),               
            conv_2_block(base_dim, base_dim*2),      
            conv_3_block(base_dim*2, base_dim*4),    
            conv_3_block(base_dim*4, base_dim*8),    
            conv_3_block(base_dim*8, base_dim*8),    
        )
        self.classifier = nn.Sequential(
            nn.Linear(base_dim*8*1*1, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 1000),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1000, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

# ──── 3. 설정 ───────────────────────────────────────────────────
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size    = 100
learning_rate = 2e-4
num_epochs    = 10

# ──── 4. 데이터 준비 & train/val split ──────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),
                         (0.2470,0.2435,0.2616))
])

# 전체 train+val 데이터셋
full_train_ds = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_ds = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

# 90% train / 10% val 로 분할
total_size = len(full_train_ds)
val_size   = int(total_size * 0.1)
train_size = total_size - val_size
train_ds, val_ds = random_split(full_train_ds, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=4)

# ──── 5. 모델·손실·옵티마이저·TensorBoard 준비 ─────────────────
model     = VGG16_CIFAR(base_dim=64, num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
writer    = SummaryWriter(log_dir="runs/cifar10_vgg")

# ──── 6. 학습 루프 ─────────────────────────────────────────────
global_step = 0
for epoch in range(1, num_epochs+1):
    ### (1) Train 단계
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        global_step  += 1

        if global_step % 50 == 0:
            writer.add_scalar("Train/Batch_Loss", loss.item(), global_step)

    epoch_loss = running_loss / len(train_loader)
    writer.add_scalar("Train/Epoch_Loss", epoch_loss, epoch)

    ### (2) Validation 단계
    model.eval()
    val_loss    = 0.0
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc   = val_correct / len(val_ds)

    writer.add_scalar("Val/Loss", val_loss, epoch)
    writer.add_scalar("Val/Acc",  val_acc,   epoch)

    print(f"[Epoch {epoch:02d}/{num_epochs}] "
          f"Train Loss: {epoch_loss:.4f}  Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

# ──── 7. Test(최종 평가) ────────────────────────────────────────
model.eval()
test_correct = 0
test_total   = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds   = outputs.argmax(1)
        test_correct += (preds == labels).sum().item()
        test_total   += labels.size(0)

print(f">>> Final Test Accuracy: {test_correct / test_total:.4f}")

# ──── 8. 마무리 ────────────────────────────────────────────────
writer.close()
